In [3]:
%%writefile spending_analysis_app.py

# 라이브러리 설치
import streamlit as st
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import calendar
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus


# 폰트 설정 (한글 깨짐 방지)
# macOS
#plt.rcParams['font.family'] = 'AppleGothic'
# Windows
plt.rcParams['font.family'] = 'Malgun Gothic'
# "-" 기호 깨짐 방지
plt.rcParams['axes.unicode_minus'] = False


# DB 연결
def get_engine():
    user = 
    password = 
    host = 
    db_name = 
    return create_engine(f"mysql+pymysql://{user}:{password}@{host}:3306/{db_name}")

# 기존 데이터 삭제
def save_to_db_fresh(df, engine):
    with engine.begin() as conn:
        conn.execute(text("SET FOREIGN_KEY_CHECKS = 0;"))
        conn.execute(text("TRUNCATE TABLE expenditure;"))
        conn.execute(text("TRUNCATE TABLE category;"))
        conn.execute(text("SET FOREIGN_KEY_CHECKS = 1;"))

    # 카테고리 추출 후 DB 테이블에 저장
    unique_category = pd.DataFrame({'category_name': df['category'].unique()})
    unique_category.to_sql('category', con=engine, if_exists='append', index=False)

    # ID 연결
    category_db = pd.read_sql("select * from category", engine)
    category_map = dict(zip(category_db['category_name'], category_db['category_id']))
    df['category_id']=df['category'].map(category_map)

    df_final=df[['date', 'amount', 'category_id', 'text']]

    # 컬럼명 변경 및 데이터 저장
    df_final = pd.DataFrame({
        'expenditure_date': df['date'],
        'expenditure_amount': df['amount'],
        'expenditure_category': df['category'].map(category_map),
        'expenditure_text': df['text']
    })
    df_final.to_sql('expenditure', con=engine, if_exists='append', index=False)
    return df_final


# 스트림릿 생성
st.title("💰 spending analysis project")
st.divider()  #구분선

# 분석 메뉴 생성
st.markdown("### 분석 항목 선택")
menu = st.radio(
    "",
    ["🔎 월간 지출 분석", "🔮 소비 예측"],
    horizontal=True,  # 가로 배치
    label_visibility="collapsed"  # 레이블 숨김
)
st.divider()

# 월간 지출 분석
if menu == "🔎 월간 지출 분석":
    st.markdown("### 월간 지출 분석")
    file_up = st.file_uploader("CSV 파일을 선택하세요.", type=['csv'], key="v_up")
    target_budget = st.number_input("목표 예산 설정(원)", value=400000, key="v_bg")
    st.write("\n")
    analyze_btn = st.button("▶️ 분석 시작", key="v_btn") # 분석 시작 버튼
    st.divider()
    
    if file_up and analyze_btn:
        df = pd.read_csv(file_up)
        engine = get_engine()
        df_final = save_to_db_fresh(df, engine)
        sub_tab1, sub_tab2, sub_tab3, sub_tab4 = st.tabs(["카테고리별", "주차별", "일별 히트맵", "소비 리포트"]) #탭 생성

        
        # 카테고리별 지출
        with sub_tab1:
            query = """
            SELECT c.category_name, SUM(e.expenditure_amount) as total_amount
            FROM expenditure e
            JOIN category c ON e.expenditure_category = c.category_id
            GROUP BY c.category_name;
            """
            df_pie = pd.read_sql(query, engine)
    
            target_month = pd.to_datetime(df_final['expenditure_date']).iloc[0].month
    
            # 파이 차트 생성
            fig1, ax1 = plt.subplots(figsize=(10, 7))
            if not df_pie.empty:
                ax1.pie(df_pie['total_amount'], 
                        labels=df_pie['category_name'], 
                        autopct='%1.1f%%', # 소수점 첫째 자리까지 표시
                        startangle=140,    # 그래프 시작 각도 설정
                        colors=plt.cm.Set3.colors)
                ax1.axis('equal')  # 비율 조정
                st.write("\n")
                st.subheader(f"📊 {target_month}월 카테고리별 지출")
                st.pyplot(fig1)
            else:
                st.warning("데이터를 삽입하세요.")

        
        # 주별 지출 추이
        with sub_tab2:
            query = """
            SELECT DATE_FORMAT(expenditure_date, '%%Y-%%u주') as week, SUM(expenditure_amount) as total_amount
            FROM expenditure
            GROUP BY week
            ORDER BY week;
            """
            df_weekly = pd.read_sql(query, engine)
    
            target_month = pd.to_datetime(df_final['expenditure_date']).iloc[0].month
            
            # 선 그래프 생성
            fig2, ax2 = plt.subplots(figsize=(10, 6))
            x_axis = list(range(1, len(df_weekly) + 1)) # X축 설정
            ax2.plot(x_axis, df_weekly['total_amount'], marker='o', color='tomato', linewidth=2)
            
            max_val = df_weekly['total_amount'].max()
            ax2.set_ylim(0, max_val * 1.2)  # Y축 여유 공간 설정
            
            ax2.set_xlabel('주차')
            ax2.set_ylabel('금액(원)')
            ax2.set_xticks(x_axis) # X축 눈금 정수로 고정
            ax2.grid(True, linestyle='--', alpha=0.6) #배경에 점선 추가
            
            for i, txt in enumerate(df_weekly['total_amount']):  # 각 점 위에 금액 표시
                ax2.annotate(f'{int(txt):,}원', (x_axis[i], df_weekly['total_amount'][i]), 
                textcoords="offset points", xytext=(0,10), ha='center')
            st.write("\n")
            st.subheader(f"📈 {target_month}월 주별 지출 추이")
            st.pyplot(fig2)


        # 일별 소비 히트맵
        with sub_tab3:
            query = """
            SELECT expenditure_date, SUM(expenditure_amount) as daily_total 
            FROM expenditure 
            GROUP BY expenditure_date;
            """
            df = pd.read_sql(query, engine)
    
            # 날짜 정보 추출
            df['expenditure_date'] = pd.to_datetime(df['expenditure_date'])
            year = df['expenditure_date'].dt.year.iloc[0]
            month = df['expenditure_date'].dt.month.iloc[0]
            df['day'] = df['expenditure_date'].dt.day 
            df['weekday'] = df['expenditure_date'].dt.weekday 
            
            # 특정 연도와 월의 시작 요일 찾음
            first_weekday = calendar.monthrange(year, month)[0]
            
            # 데이터가 없는 날을 0원으로 채움
            all_days = pd.DataFrame({'day': range(1, 32)})
            df_full = pd.merge(all_days, df, left_on='day', right_on='df.expenditure_date.dt.day' 
                               if False else 'day', how='left').fillna(0)
    
            # 히트맵 생성
            calendar_grid = np.zeros((6, 7))
            day_labels = np.zeros((6, 7))
            
            for d in range(1, 32):  # 위치에 맞게 금액 및 날짜 배치
                day_data = df_full[df_full['day'] == d]
                amount = day_data['daily_total'].values[0] if not day_data.empty else 0
            
                idx = d + first_weekday - 1
                row = idx // 7
                col = idx % 7
            
                if row < 6:  # 지출 금액 저장
                    calendar_grid[row, col] = amount
                    day_labels[row, col] = d
            
            fig3, ax3 = plt.subplots(figsize=(12, 8))
            im = ax3.imshow(calendar_grid, cmap='YlOrRd', vmin=1, aspect='equal') 
            
            ax3.set_xticks(np.arange(7))
            ax3.set_xticklabels(['월', '화', '수', '목', '금', '토', '일'])
            ax3.set_yticks(np.arange(6))
            ax3.set_yticklabels([f'{i+1}주차' for i in range(6)])
            
            # 히트맵 위에 텍스트(날짜, 금액) 표시
            for i in range(6):
                for j in range(7):
                    day_num = int(day_labels[i, j])
                    if day_num > 0:  # 날짜가 있는 칸만 작성
                        ax3.text(j-0.4, i-0.3, str(day_num), va='top', ha='left', fontsize=10, color='gray')
                        amount = calendar_grid[i, j]
                        if amount > 0:  # 돈을 쓴 날만 금액 표시
                            color = 'white' if amount > calendar_grid.max() * 0.6 else 'black'
                            ax3.text(j, i, f'{int(amount):,}원', ha='center', va='center', fontweight='bold', color=color)
            
            fig3.colorbar(im, label='지출 금액 (원)', shrink=0.8)
            fig3.tight_layout()
            st.write("\n")
            st.subheader(f"📅 {month}월 소비 히트맵")
            st.pyplot(fig3)

        
        # 소비 목표 관리
        with sub_tab4:
            query = "SELECT SUM(expenditure_amount) as total FROM expenditure"
            df_total = pd.read_sql(query, engine)
            current_spend = df_total['total'].iloc[0] if df_total['total'].iloc[0] else 0  # 금액 추출
            
            target_date = pd.to_datetime(df['expenditure_date']).iloc[0]
            year, month = target_date.year, target_date.month  # 날짜 정보 추출

            st.write("\n")
            st.subheader(f"📋 {year}년 {month}월 소비 리포트")
            st.write("\n\n")
            
            col1, col2 = st.columns(2)  # 가로로 나열
            with col1:
                st.metric("목표 예산", f"{target_budget:,}원")
            with col2:
                st.metric("총 지출액", f"{int(current_spend):,}원")
            st.write("\n\n")
            
            # 목표 예산 초과한 경우
            if current_spend > target_budget:
                over_amount = current_spend - target_budget
                over_ratio = (over_amount / target_budget) * 100
                st.error(f"🚨 **[경고] 목표 예산을 초과했습니다!**")
                st.markdown(f"""
                - <span style="font-size: 18px;">**초과 금액**: &nbsp;{int(over_amount):,}원</span>
                - <span style="font-size: 18px;">**초과 비율**: &nbsp;{over_ratio:.2f}%</span>
                """, unsafe_allow_html=True)  # 글씨 크기 조절, &nbsp; 공백
                
            # 목표 예산을 초과하지 않은 경우
            else:
                remaining_budget = target_budget - current_spend # 남은 금액
                spend_ratio = (current_spend / target_budget) * 100
                st.success(f"✅ **목표 범위 내에서 소비하셨습니다.**")
                st.markdown(f"""
                - <span style="font-size: 18px;">**남은 여유 자금**: &nbsp;{int(remaining_budget):,}원</span>
                - <span style="font-size: 18px;">**예산 사용률**: &nbsp;{spend_ratio:.1f}%</span>  
                """, unsafe_allow_html=True)

    elif file_up and not analyze_btn:
        st.info("'분석 시작' 버튼을 눌러 분석 결과를 확인해주세요.")


# 소비 예측
elif menu == "🔮 소비 예측":
    st.markdown("### 소비 예측")
    file_pred = st.file_uploader("CSV 파일을 선택하세요", type=['csv'], key="p_up")
    budget_pred = st.number_input("목표 예산 설정(원)", value=400000, key="p_bg")
    st.write("\n")
    predict_btn = st.button("▶️ 예측 시작", key="p_btn")  # 예측 시작 버
    st.divider()

    if file_pred and predict_btn:
        df_now = pd.read_csv(file_pred)
        # 날짜 정보 추출
        df_now['date'] = pd.to_datetime(df_now['date']) 
        today = df_now['date'].max()
        curr_year = today.year
        curr_month = today.month
        curr_day = today.day
        _, last_day = calendar.monthrange(curr_year, curr_month)

        spend_now = df_now['amount'].sum() # 지출 합계
        daily_avg = spend_now / curr_day # 일평균
        predicted_total = daily_avg * last_day # 예상 총 지출

        st.write("\n")
        st.subheader(f"📋 {curr_year}년 {curr_month}월 소비 예측")
        st.write("\n\n")
        
        col1, col2, col3 = st.columns(3)  # 가로로 나열
        with col1:
            st.metric("목표 예산", f"{budget_pred:,}원")
        with col2:
            st.metric("현재까지 누적 지출", f"{int(spend_now):,}원")
        with col3:
            st.metric("월말 예상 총 지출", f"{int(predicted_total):,}원")
        st.write("\n\n")

        # 예상 총 지출이 목표 예산을 초과하는 경우
        if predicted_total > budget_pred:
            over_amount = predicted_total - budget_pred
            st.error(f"🚨 [경고] 목표 예산을 약 {int(over_amount):,}원 초과할 것으로 예상됩니다!")
        
            remaining_days = last_day - curr_day # 남은 날짜 계산
            safe_daily = (budget_pred - spend_now) / remaining_days if remaining_days > 0 else 0  # 남은 기간 하루 권장 지출액 계산
            st.markdown(f"""
            - <span style="font-size: 18px;">**하루 권장 지출액**: &nbsp;{int(safe_daily):,}원</span>
            """, unsafe_allow_html=True)

        # 예상 총 지출이 목표 예산을 초과하지 않는 경우
        else:
            st.success("✅ 목표 범위 내 소비 중입니다.")
            
    elif file_pred and not predict_btn:
        st.info("'예측 시작' 버튼을 눌러 예측 결과를 확인하세요.")

Overwriting spending_analysis_app.py


In [4]:
# 터미널에서 파일이 저장된 경로로 이동후 실행
# streamlit run spending_analysis_app.py